# 20 — Practice: Module 4 — Aggregation & Transformation

30 interview-style questions on groupby, pivot, merge, window functions,
and reshaping — the core of data engineering interviews.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

# Sales dataset
sales = pd.DataFrame({
    'order_id':    range(1, n + 1),
    'customer_id': np.random.randint(1, 100, n),
    'salesperson': np.random.choice(['Alice', 'Bob', 'Carol', 'Dave', 'Eve'], n),
    'region':      np.random.choice(['North', 'South', 'East', 'West'], n),
    'category':    np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'revenue':     np.round(np.random.uniform(500, 80000, n), 2),
    'units':       np.random.randint(1, 50, n),
    'year':        np.random.choice([2022, 2023], n),
    'quarter':     np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], n),
    'date':        pd.date_range('2022-01-01', periods=n, freq='D')[:n]
})

# Employee dataset
employees = pd.DataFrame({
    'emp_id':     range(1001, 1101),
    'name':       [f'Emp_{i}' for i in range(100)],
    'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing'], 100),
    'salary':     np.random.randint(40000, 180000, 100),
    'rating':     np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], 100, p=[0.05, 0.1, 0.25, 0.4, 0.2]),
    'experience': np.random.randint(1, 20, 100),
    'dept_id':    np.random.choice([101, 102, 103, 104], 100)
})

departments = pd.DataFrame({
    'dept_id':   [101, 102, 103, 105],  # 104 missing, 105 extra
    'dept_name': ['Engineering', 'Sales', 'HR', 'Marketing'],
    'location':  ['Bangalore', 'Mumbai', 'Delhi', 'Pune']
})

print(f"sales: {sales.shape}  employees: {employees.shape}  departments: {departments.shape}")

---
### Q1: Total and average revenue by region and category

In [ ]:
result = sales.groupby(['region', 'category']).agg(
    orders=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean')
).round(0)
print(result)

---
### Q2: Add a column with each salesperson's total revenue (transform)

In [ ]:
sales['salesperson_total'] = sales.groupby('salesperson')['revenue'].transform('sum').round(0)
print(sales[['order_id', 'salesperson', 'revenue', 'salesperson_total']].head(5))

---
### Q3: Find the top salesperson by revenue per region

In [ ]:
top_by_region = (
    sales.groupby(['region', 'salesperson'])['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue', ascending=False)
    .groupby('region', group_keys=False)
    .apply(lambda g: g.nlargest(1, 'revenue'))
)
print(top_by_region.round(0))

---
### Q4: Pivot — revenue by salesperson (rows) × category (columns)

In [ ]:
pt = pd.pivot_table(
    sales, values='revenue',
    index='salesperson', columns='category',
    aggfunc='sum', margins=True, fill_value=0
).round(0)
print(pt)

---
### Q5: Crosstab showing order counts by region × year

In [ ]:
ct = pd.crosstab(sales['region'], sales['year'], margins=True)
print(ct)

---
### Q6: Melt a wide dataframe — Q1/Q2/Q3/Q4 revenue columns → long format

In [ ]:
wide = pd.DataFrame({
    'salesperson': ['Alice', 'Bob', 'Carol'],
    'Q1_rev': [100000, 90000, 120000],
    'Q2_rev': [110000, 95000, 130000],
    'Q3_rev': [105000, 88000, 125000],
    'Q4_rev': [120000, 100000, 140000],
})

long = wide.melt(id_vars='salesperson', var_name='quarter', value_name='revenue')
long['quarter'] = long['quarter'].str.replace('_rev', '')
print(long)

---
### Q7: Inner join employees to departments

In [ ]:
merged = pd.merge(employees, departments, on='dept_id', how='inner')
print(f"Employees: {len(employees)}  Merged (inner): {len(merged)}")
print(merged[['emp_id', 'name', 'dept_id', 'dept_name', 'salary']].head(5))

---
### Q8: Find employees with no matching department (anti-join)

In [ ]:
no_dept = (
    pd.merge(employees, departments, on='dept_id', how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
)
print(f"Employees without matching dept: {len(no_dept)}")
print(no_dept[['emp_id', 'name', 'dept_id']].head())

---
### Q9: Get each employee's salary relative to their department average

In [ ]:
employees['dept_avg_salary'] = employees.groupby('department')['salary'].transform('mean').round(0)
employees['salary_diff'] = (employees['salary'] - employees['dept_avg_salary']).round(0)

print(employees[['name', 'department', 'salary', 'dept_avg_salary', 'salary_diff']].head(10))

---
### Q10: Rank employees by salary within department

In [ ]:
employees['dept_rank'] = (
    employees.groupby('department')['salary']
    .rank(ascending=False, method='dense')
    .astype(int)
)
print(employees[['name', 'department', 'salary', 'dept_rank']]
      .sort_values(['department', 'dept_rank'])
      .head(12))

---
### Q11: 7-day rolling average revenue

In [ ]:
daily_sales = sales.set_index('date').sort_index()['revenue'].resample('D').sum()
daily_sales_7d_avg = daily_sales.rolling(7, min_periods=1).mean().round(0)
print(pd.DataFrame({'daily': daily_sales, '7d_avg': daily_sales_7d_avg}).head(10))

---
### Q12: YTD cumulative revenue

In [ ]:
ytd = daily_sales.expanding().sum()
print(ytd.head(10))

---
### Q13: Compute revenue lag (yesterday's revenue) and day-over-day % change

In [ ]:
ts = daily_sales.to_frame('revenue')
ts['lag1'] = ts['revenue'].shift(1)
ts['dod_pct'] = (ts['revenue'].pct_change(1) * 100).round(2)
print(ts.dropna().head(7))

---
### Q14: Keep only departments where mean rating > 3.5

In [ ]:
high_rated = employees.groupby('department').filter(
    lambda g: g['rating'].mean() > 3.5
)
print(high_rated['department'].value_counts())

---
### Q15: stack/unstack — average salary by department × experience band

In [ ]:
employees['exp_band'] = pd.cut(employees['experience'],
                                bins=[0, 5, 10, 20],
                                labels=['Junior', 'Mid', 'Senior'])

pivot = (
    employees.groupby(['department', 'exp_band'], observed=True)['salary']
    .mean()
    .round(0)
    .unstack('exp_band')
)
print(pivot)

---
### Q16: Self-join to find employee pairs in the same department

In [ ]:
pairs = pd.merge(
    employees[['emp_id', 'name', 'department']],
    employees[['emp_id', 'name', 'department']],
    on='department',
    suffixes=('_a', '_b')
)
# Remove self-pairs and duplicates
pairs = pairs[pairs['emp_id_a'] < pairs['emp_id_b']]
print(f"Pairs in same department: {len(pairs):,}")
print(pairs[['name_a', 'name_b', 'department']].head(5))

---
### Q17: Cross join products × regions to generate all combinations

In [ ]:
products = pd.DataFrame({'product': ['Laptop', 'Phone', 'Tablet']})
regions  = pd.DataFrame({'region': ['North', 'South', 'East', 'West']})

all_combos = pd.merge(products, regions, how='cross')
print(f"Combinations: {len(all_combos)}")
print(all_combos)

---
### Q18: concat 2022 and 2023 sales, add year label

In [ ]:
sales_2022 = sales[sales['year'] == 2022].copy()
sales_2023 = sales[sales['year'] == 2023].copy()

combined = pd.concat(
    [sales_2022, sales_2023],
    keys=['2022', '2023'],
    names=['year_label', None]
)
print(combined.index.get_level_values('year_label').value_counts())

---
### Q19: Get top 3 orders per region (apply with nlargest)

In [ ]:
top3 = (
    sales
    .groupby('region', group_keys=False)
    .apply(lambda g: g.nlargest(3, 'revenue'))
    [['order_id', 'region', 'salesperson', 'revenue']]
    .reset_index(drop=True)
)
print(top3)

---
### Q20: Wide-to-long panel data with wide_to_long()

In [ ]:
panel = pd.DataFrame({
    'company':     ['Acme', 'Beta', 'Gamma'],
    'revenue_2021': [500, 300, 450],
    'revenue_2022': [550, 320, 480],
    'revenue_2023': [600, 350, 510],
    'profit_2021':  [50, 30, 45],
    'profit_2022':  [55, 32, 48],
    'profit_2023':  [60, 35, 51],
})

long = pd.wide_to_long(
    panel, stubnames=['revenue', 'profit'],
    i='company', j='year', sep='_'
).reset_index()
print(long)

---
### Q21: Compute salary z-score within each department

In [ ]:
employees['salary_zscore'] = (
    employees.groupby('department')['salary']
    .transform(lambda x: (x - x.mean()) / x.std())
    .round(2)
)
print(employees[['name', 'department', 'salary', 'salary_zscore']].head(10))

---
### Q22: Revenue percentile rank across all orders

In [ ]:
sales['revenue_pct_rank'] = (sales['revenue'].rank(pct=True) * 100).round(1)
print("Top 10% orders:")
print(sales[sales['revenue_pct_rank'] >= 90][['order_id', 'salesperson', 'revenue', 'revenue_pct_rank']].head())

---
### Q23: Exponential weighted average of daily revenue (span=7)

In [ ]:
ts2 = daily_sales.to_frame('revenue')
ts2['ewm_7'] = ts2['revenue'].ewm(span=7, adjust=False).mean().round(0)
print(ts2.head(10))

---
### Q24: validate='m:1' on employee-department merge

In [ ]:
try:
    result = pd.merge(employees, departments, on='dept_id', how='inner', validate='m:1')
    print(f"m:1 validate passed: {result.shape}")
except Exception as e:
    print(f"Validation failed: {e}")

---
### Q25: Compute headcount, total payroll, and avg salary per dept using named agg

In [ ]:
payroll = employees.groupby('department').agg(
    headcount     = ('emp_id',   'count'),
    total_payroll = ('salary',   'sum'),
    avg_salary    = ('salary',   'mean'),
    median_salary = ('salary',   'median'),
    avg_rating    = ('rating',   'mean'),
    avg_exp       = ('experience','mean')
).round(0)
print(payroll)

---
## Module 4 Summary

| Topic | Key Pattern |
|-------|-------------|
| Named agg | `groupby().agg(col=('src', func))` |
| Group-level feature | `groupby().transform('mean')` |
| Top-N per group | `.groupby().apply(lambda g: g.nlargest(N, col))` |
| Pivot table | `pd.pivot_table(..., margins=True)` |
| Melt | `melt(id_vars, var_name, value_name)` |
| Inner join | `pd.merge(df1, df2, on='key', how='inner')` |
| Anti-join | merge with `indicator=True` → filter `left_only` |
| Self-join | merge df with itself on different key alias |
| Rolling avg | `.rolling(7, min_periods=1).mean()` |
| Lag features | `.shift(1)`, `.pct_change(1)`, `.diff(1)` |
| Rank | `.rank(ascending=False, method='dense')` |